In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

from aeon.classification.interval_based import RSTSF
from tscglue.interval_models import RSTSFRandom, RSTSFUnsupervised
from tscglue import data_loader

In [2]:
# Smaller UCR datasets
DATASETS = [
    "ArrowHead",      # 36 train,  175 test, len=251, 3 classes
    "Coffee",         # 28 train,   28 test, len=286, 2 classes
    "BME",            # 30 train,  150 test, len=128, 3 classes
    "ECG200",         # 100 train, 100 test, len=96,  2 classes
    "CBF",            # 30 train,  900 test, len=128, 3 classes
    "Beef",           # 30 train,   30 test, len=470, 5 classes
    "OliveOil",       # 30 train,   30 test, len=570, 4 classes
    "Wine",           # 57 train,   54 test, len=234, 2 classes
    "Plane",          # 105 train, 105 test, len=144, 7 classes
    "GunPoint",       # 50 train,  150 test, len=150, 2 classes
    "FaceFour",       # 24 train,   88 test, len=350, 4 classes
    "Fish",           # 175 train, 175 test, len=463, 7 classes
    "Trace",          # 100 train, 100 test, len=275, 4 classes
    "SyntheticControl", # 300 train, 300 test, len=60, 6 classes
    "ItalyPowerDemand", # 67 train,  1029 test, len=24, 2 classes
]

CLASSIFIERS = [
    ("RSTSF",              lambda: RSTSF(n_estimators=200, n_intervals=50, random_state=42, n_jobs=-1)),
    ("RSTSF-Random",       lambda: RSTSFRandom(n_estimators=200, n_intervals=600, random_state=42, n_jobs=-1)),
    ("RSTSF-Unsupervised", lambda: RSTSFUnsupervised(n_estimators=200, n_intervals=50, random_state=42, n_jobs=-1)),
]

In [ ]:
results = []

for dataset in DATASETS:
    X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, fold=5)
    print(f"\n{dataset}: train={X_train.shape}, test={X_test.shape}")

    for clf_name, clf_fn in CLASSIFIERS:
        clf = clf_fn()
        t0 = time.perf_counter()
        clf.fit(X_train, y_train)
        fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        acc = accuracy_score(y_test, clf.predict(X_test))
        pred_time = time.perf_counter() - t0

        results.append({
            "dataset": dataset,
            "classifier": clf_name,
            "accuracy": acc,
            "fit_s": round(fit_time, 1),
            "predict_s": round(pred_time, 2),
        })
        print(f"  {clf_name:22s}  acc={acc:.4f}  fit={fit_time:.1f}s")


ArrowHead: train=(36, 1, 251), test=(175, 1, 251)
  RSTSF                   acc=0.7371  fit=3.7s
  RSTSF-Random            acc=0.7771  fit=4.7s
  RSTSF-Unsupervised      acc=0.7086  fit=28.8s

Coffee: train=(28, 1, 286), test=(28, 1, 286)
  RSTSF                   acc=1.0000  fit=2.6s
  RSTSF-Random            acc=1.0000  fit=4.3s
  RSTSF-Unsupervised      acc=0.9643  fit=34.1s

BME: train=(30, 1, 128), test=(150, 1, 128)
  RSTSF                   acc=1.0000  fit=2.2s
  RSTSF-Random            acc=1.0000  fit=4.9s
  RSTSF-Unsupervised      acc=0.9467  fit=14.6s

ECG200: train=(100, 1, 96), test=(100, 1, 96)
  RSTSF                   acc=0.9000  fit=3.6s
  RSTSF-Random            acc=0.9100  fit=7.5s
  RSTSF-Unsupervised      acc=0.8700  fit=17.1s

CBF: train=(30, 1, 128), test=(900, 1, 128)
  RSTSF                   acc=0.9933  fit=2.4s
  RSTSF-Random            acc=0.9956  fit=5.0s
  RSTSF-Unsupervised      acc=0.9267  fit=14.4s

Beef: train=(30, 1, 470), test=(30, 1, 470)
  RSTSF   

In [4]:
df = pd.DataFrame(results)

# Accuracy pivot
acc_table = df.pivot(index="dataset", columns="classifier", values="accuracy")
acc_table = acc_table[["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised"]]
print("=== Accuracy ===")
print(acc_table.to_string(float_format="{:.4f}".format))

print()

# Fit time pivot
time_table = df.pivot(index="dataset", columns="classifier", values="fit_s")
time_table = time_table[["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised"]]
print("=== Fit time (s) ===")
print(time_table.to_string(float_format="{:.1f}".format))

=== Accuracy ===
classifier        RSTSF  RSTSF-Random  RSTSF-Unsupervised
dataset                                                  
ArrowHead        0.7371        0.7771              0.7086
BME              1.0000        1.0000              0.9467
Beef             0.8000        0.8333              0.7000
CBF              0.9933        0.9956              0.9267
Coffee           1.0000        1.0000              0.9643
ECG200           0.9000        0.9100              0.8700
FaceFour         1.0000        0.9659              0.6818
Fish             0.9257        0.9371              0.9029
GunPoint         0.9800        0.9933              0.9867
ItalyPowerDemand 0.9660        0.9718              0.9611
OliveOil         0.9333        0.9333              0.6667
Plane            1.0000        1.0000              0.9714
SyntheticControl 0.9933        0.9933              0.9300
Trace            1.0000        1.0000              0.9800
Wine             0.8704        0.8148              0.70

In [5]:
# Mean accuracy per classifier
print("Mean accuracy:")
print(df.groupby("classifier")["accuracy"].mean().reindex(["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised"]).to_string(float_format="{:.4f}".format))

print()
print("Mean fit time (s):")
print(df.groupby("classifier")["fit_s"].mean().reindex(["RSTSF", "RSTSF-Random", "RSTSF-Unsupervised"]).to_string(float_format="{:.1f}".format))

Mean accuracy:
classifier
RSTSF                0.9399
RSTSF-Random         0.9417
RSTSF-Unsupervised   0.8600

Mean fit time (s):
classifier
RSTSF                 4.1
RSTSF-Random          6.8
RSTSF-Unsupervised   35.9
